# CHELSA V2.1 — normal climática do Brasil (1991-2020)

Baixa e recorta para o Brasil (bbox retangular) as variáveis mensais `tas`, `pet` e `pr` do
[CHELSA V2.1](https://chelsa-climate.org/), calcula a normal de 30 anos (1991-2020) por mês e
monta uma única imagem GeoTIFF multibanda (36 bandas: 3 variáveis x 12 meses), pronta para
subir manualmente como asset no Google Earth Engine.

PET tem meses faltantes na fonte; a média usa apenas os anos disponíveis em cada mês.

## 1. Baixar e recortar os arquivos mensais

Gera o manifesto de disponibilidade e recorta cada arquivo mensal para o bbox do Brasil (sem baixar os GeoTIFFs globais inteiros).

In [ ]:
import sys
sys.path.insert(0, ".")

import baixar_recortar
baixar_recortar.main()

## 2. Conferir o manifesto de disponibilidade

In [ ]:
import pandas as pd

manifesto = pd.read_csv(baixar_recortar.MANIFESTO_PATH)
manifesto.groupby("variavel")["encontrado"].value_counts().unstack(fill_value=0)

## 3. Calcular a normal 1991-2020 e montar a imagem multibanda

In [ ]:
import gerar_normal_multibanda
gerar_normal_multibanda.main()

## 4. Conferência visual de uma banda por variável

In [ ]:
import rasterio
import matplotlib.pyplot as plt

with rasterio.open(gerar_normal_multibanda.SAIDA_PATH) as src:
    fig, eixos = plt.subplots(1, 3, figsize=(15, 5))
    for eixo, nome_banda in zip(eixos, ["tas_01", "pet_01", "pr_01"]):
        indice = src.descriptions.index(nome_banda) + 1
        dado = src.read(indice)
        im = eixo.imshow(dado, cmap="viridis")
        eixo.set_title(nome_banda)
        plt.colorbar(im, ax=eixo, shrink=0.7)
    plt.tight_layout()
    plt.show()

## 5. Checagem de sanidade da imagem final

In [ ]:
with rasterio.open(gerar_normal_multibanda.SAIDA_PATH) as src:
    print("Arquivo:", gerar_normal_multibanda.SAIDA_PATH)
    print("Bandas:", src.count)
    print("Dimensões:", src.width, "x", src.height)
    print("Dtype:", src.dtypes[0])
    print("CRS:", src.crs)
    print("Nomes das bandas:", list(src.descriptions))

## 6. Upload manual para o Google Earth Engine

Este notebook não faz upload automático. Para subir a imagem final como asset:

1. Enviar o arquivo `dados_chelsa/normal_1991_2020/chelsa_brasil_normal_1991_2020.tif` para um bucket do Google Cloud Storage.
2. Rodar:

```powershell
earthengine upload image --asset_id=projects/SEU_PROJETO/assets/chelsa_brasil_normal_1991_2020 gs://SEU_BUCKET/chelsa_brasil_normal_1991_2020.tif
```